# Seminar 11: Tokenization, Encoder vs Decoder, and Hugging Face Workflow

Today is a practical Hugging Face lab.

Goals:
- inspect subword tokenization instead of guessing it,
- understand `input_ids`, `attention_mask`, and special tokens,
- use DistilBERT for masked-token prediction,
- use DistilGPT2 for next-token prediction and generation,
- summarize the encoder vs decoder difference from evidence you produced.

Expected rhythm: 5 medium board tasks, about 90 minutes total.


## 0. Setup

If you run this in Colab and `transformers` is missing, uncomment the install line first.


In [ ]:
# If needed in Colab, uncomment:
# %pip install transformers sentencepiece accelerate -q

import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM

pd.set_option('display.max_colwidth', 120)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


### Provided Model Loading

This is not an exercise. Loading models is mechanical, and the interesting work starts after the tokenizer/model objects exist.

Important ideas:
- tokenizer and model must come from the same model family,
- DistilBERT is an encoder-style masked language model,
- DistilGPT2 is a decoder-style causal language model,
- GPT2 has no padding token by default, so we reuse the EOS token for batching.


In [ ]:
MASKED_MODEL_NAME = 'distilbert-base-uncased'
CAUSAL_MODEL_NAME = 'distilgpt2'

bert_tok = AutoTokenizer.from_pretrained(MASKED_MODEL_NAME)
gpt_tok = AutoTokenizer.from_pretrained(CAUSAL_MODEL_NAME)

bert_mlm = AutoModelForMaskedLM.from_pretrained(MASKED_MODEL_NAME).to(device)
gpt_lm = AutoModelForCausalLM.from_pretrained(CAUSAL_MODEL_NAME).to(device)

bert_mlm.eval()
gpt_lm.eval()

gpt_tok.pad_token = gpt_tok.eos_token

print('Loaded:', MASKED_MODEL_NAME)
print('Loaded:', CAUSAL_MODEL_NAME)


## 1. Exercise 1: Tokenizer Detective

Subword tokenization is easier to understand by inspecting examples.

Task:
- implement `compare_tokenizers(text)`,
- compare BERT-style and GPT-style tokenization for the same text,
- return a table with token strings, token ids, and token counts.

Function contract:
- `compare_tokenizers(text)` receives one Python string,
- it returns a `pandas.DataFrame`,
- required columns: `tokenizer`, `text`, `tokens`, `token_ids`, `num_tokens`.

Useful tokenizer methods:
- `tokenizer.tokenize(text)` returns token strings,
- `tokenizer.convert_tokens_to_ids(tokens)` converts token strings to integer ids.


In [ ]:
comparison_texts = [
    'Transformers changed natural language processing.',
    'unbelievable',
    'ChatGPT-like systems are everywhere now!',
    'bioinformatics-friendly tokenization?',
    'The     spacing is strange here.',
]


def compare_tokenizers(text):
    rows = []

    # Add one row for BERT.
    # Add one row for GPT.
    # Each row should store tokens, token ids, and number of tokens.

    return pd.DataFrame(rows)


tokenizer_comparison = compare_tokenizers(comparison_texts[2])
tokenizer_comparison


### Checks (Exercise 1)

In [ ]:
assert isinstance(tokenizer_comparison, pd.DataFrame)

required_columns = ['tokenizer', 'text', 'tokens', 'token_ids', 'num_tokens']
for column in required_columns:
    assert column in tokenizer_comparison.columns

assert len(tokenizer_comparison) == 2
assert 'DistilBERT' in tokenizer_comparison['tokenizer'].values
assert 'DistilGPT2' in tokenizer_comparison['tokenizer'].values

for row_index in range(len(tokenizer_comparison)):
    tokens = tokenizer_comparison.loc[row_index, 'tokens']
    token_ids = tokenizer_comparison.loc[row_index, 'token_ids']
    num_tokens = tokenizer_comparison.loc[row_index, 'num_tokens']
    assert len(tokens) == len(token_ids)
    assert num_tokens == len(tokens)
    assert num_tokens > 0

print('Exercise 1 passed.')


## 2. Exercise 2: From Text to Model Inputs

Token strings are only the beginning. Models receive tensors.

Task:
- tokenize a small batch with padding,
- inspect `input_ids`, `attention_mask`, and tokens position by position,
- compare how BERT and GPT inputs look.

Function contract:
- `show_model_inputs(tokenizer, texts)` receives a tokenizer and a list of strings,
- it returns a table with columns: `text_index`, `position`, `token`, `token_id`, `attention_mask`,
- it should use padding, because texts have different lengths.

Useful tokenizer call:
- `tokenizer(texts, return_tensors='pt', padding=True)` returns a dictionary with `input_ids` and `attention_mask`.

Useful tokenizer method:
- `tokenizer.convert_ids_to_tokens(ids)` maps ids back to token strings.


In [ ]:
batch_texts = [
    'Deep learning is useful.',
    'Tokenizers turn text into numbers.',
]


def show_model_inputs(tokenizer, texts):
    rows = []

    # Tokenize texts with padding.
    # Loop over each text and each token position.
    # Store token, token id, and attention mask value in rows.

    return pd.DataFrame(rows)


bert_input_table = show_model_inputs(bert_tok, batch_texts)
gpt_input_table = show_model_inputs(gpt_tok, batch_texts)

bert_input_table


### Checks (Exercise 2)

In [ ]:
for table in [bert_input_table, gpt_input_table]:
    assert isinstance(table, pd.DataFrame)
    required_columns = ['text_index', 'position', 'token', 'token_id', 'attention_mask']
    for column in required_columns:
        assert column in table.columns
    assert len(table) > 0
    assert table['text_index'].nunique() == len(batch_texts)
    assert table['attention_mask'].isin([0, 1]).all()

print('Exercise 2 passed.')


## 3. Exercise 3: Masked Language Modeling With BERT

BERT-style masked language models predict a missing token using both left and right context.

Task:
- implement `predict_masked_token(text, top_k=5)`,
- find the `[MASK]` position,
- run DistilBERT,
- return top predictions with probabilities.

Function contract:
- `text` must contain exactly one mask token: `bert_tok.mask_token`,
- output columns: `rank`, `token`, `token_id`, `probability`,
- probabilities should be sorted from largest to smallest.

Useful functions:
- `bert_tok(text, return_tensors='pt')` tokenizes one string,
- `bert_mlm(**inputs).logits` returns logits of shape `[batch, sequence_length, vocab_size]`,
- `F.softmax(logits, dim=-1)` converts logits to probabilities,
- `torch.topk(probabilities, k=top_k)` returns top values and ids.


In [ ]:
masked_examples = [
    'Paris is the [MASK] of France.',
    'The programmer fixed the [MASK].',
    'The bank is near the [MASK].',
]


def predict_masked_token(text, top_k=5):
    rows = []

    # Tokenize text and run bert_mlm.
    # Find the mask position.
    # Take probabilities at this position.
    # Store top_k predictions in rows.

    return pd.DataFrame(rows)


mask_predictions = predict_masked_token(masked_examples[0], top_k=5)
mask_predictions


### Checks (Exercise 3)

In [ ]:
assert isinstance(mask_predictions, pd.DataFrame)
required_columns = ['rank', 'token', 'token_id', 'probability']
for column in required_columns:
    assert column in mask_predictions.columns

assert len(mask_predictions) == 5
assert mask_predictions['probability'].between(0, 1).all()

probabilities = mask_predictions['probability'].tolist()
for i in range(len(probabilities) - 1):
    assert probabilities[i] >= probabilities[i + 1]

print('Exercise 3 passed.')


## 4. Exercise 4: GPT Next Tokens and Temperature

GPT-style causal language models predict the next token from left context only.

Task A:
- implement `predict_next_tokens(prompt, top_k=5)`,
- inspect the most likely next tokens after a prompt.

Task B:
- use `generate_text(prompt, temperature)` to compare low and high temperature generation.

Function contract for `predict_next_tokens`:
- input: one prompt string,
- output columns: `rank`, `token`, `token_id`, `probability`,
- predictions should come from the final position in the prompt.

Useful generation arguments:
- `max_new_tokens` controls length,
- `do_sample=True` enables sampling,
- lower `temperature` is more conservative,
- higher `temperature` is more diverse but less stable.


In [ ]:
prompt = 'Deep learning is useful because'


def predict_next_tokens(prompt, top_k=5):
    rows = []

    # Tokenize prompt and run gpt_lm.
    # Take logits from the last prompt position.
    # Convert logits to probabilities.
    # Store top_k next-token candidates.

    return pd.DataFrame(rows)


def generate_text(prompt, temperature=0.8, max_new_tokens=40):
    inputs = gpt_tok(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output_ids = gpt_lm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=gpt_tok.eos_token_id,
        )

    text = gpt_tok.decode(output_ids[0], skip_special_tokens=True)
    return text


next_token_predictions = predict_next_tokens(prompt, top_k=5)
low_temperature_text = None
high_temperature_text = None

next_token_predictions


### Checks (Exercise 4)

In [ ]:
assert isinstance(next_token_predictions, pd.DataFrame)
required_columns = ['rank', 'token', 'token_id', 'probability']
for column in required_columns:
    assert column in next_token_predictions.columns

assert len(next_token_predictions) == 5
assert next_token_predictions['probability'].between(0, 1).all()

probabilities = next_token_predictions['probability'].tolist()
for i in range(len(probabilities) - 1):
    assert probabilities[i] >= probabilities[i + 1]

assert isinstance(low_temperature_text, str)
assert isinstance(high_temperature_text, str)
assert len(low_temperature_text) > len(prompt)
assert len(high_temperature_text) > len(prompt)

print('Exercise 4 passed.')


## 5. Exercise 5: Encoder vs Decoder Evidence Table

Now summarize what you observed, not just what the lecture said.

Task:
- build a small comparison table for DistilBERT and DistilGPT2,
- base the rows on experiments above.

Required columns:
- `property`, `DistilBERT`, `DistilGPT2`.

Suggested rows:
- model family,
- context direction,
- tokenization observation,
- special tokens,
- masked-token prediction,
- next-token generation,
- typical use.


In [ ]:
comparison_rows = []

# Fill comparison_rows with evidence-based observations.
# Each row should be a dictionary with keys:
# property, DistilBERT, DistilGPT2

encoder_decoder_table = pd.DataFrame(comparison_rows)
encoder_decoder_table


### Checks (Exercise 5)

In [ ]:
assert isinstance(encoder_decoder_table, pd.DataFrame)
required_columns = ['property', 'DistilBERT', 'DistilGPT2']
for column in required_columns:
    assert column in encoder_decoder_table.columns

assert len(encoder_decoder_table) >= 5

for column in required_columns:
    assert encoder_decoder_table[column].isna().sum() == 0

print('Exercise 5 passed.')


## 6. Wrap-Up Questions

1. Why are subword tokenizers useful for rare or strange words?
2. What did BERT and GPT tokenizers represent differently?
3. What does `attention_mask = 0` mean in a padded batch?
4. Why can BERT use both left and right context for `[MASK]`?
5. Why is GPT generation sensitive to temperature?
6. When would you choose an encoder-style model, and when would you choose a decoder-style model?
